In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import pennylane as qml
import numpy as np
import os

In [ ]:

# Import your original data loader
from seqnn_dataLoader import DataLoader as OriginalDataLoader

In [ ]:
# =============================================================================
# 1. QUANTUM DEVICE & CIRCUIT DEFINITIONS
# =============================================================================

# Define the device. 
# Use "lightning.qubit" for fast CPU simulation.
# Use "lightning.gpu" if you have an NVIDIA GPU and cuQuantum installed.
N_QUBITS = 12
dev = qml.device("lightning.qubit", wires=N_QUBITS)

In [ ]:

def apply_u3(theta, phi, lam, target):
    """
    Exact reproduction of the custom 'u3' gate from the original notebook:
    rz(lam) -> rx(pi/2) -> rz(theta) -> rx(-pi/2) -> rz(phi)
    """
    qml.RZ(lam, wires=target)
    qml.RX(np.pi/2, wires=target)
    qml.RZ(theta, wires=target)
    qml.RX(-np.pi/2, wires=target)
    qml.RZ(phi, wires=target)

def apply_controlled_u3(theta, phi, lam, target, controls, control_values):
    """
    Applies U3 gate controlled by specific bit states of control wires.
    Matches TFQ's: cirq.rz(lam).on(target).controlled_by(..., control_values=...)
    """
    # PennyLane's ctrl transforms the op to be controlled
    # We wrap the U3 sequence in a single function
    def u3_op():
        apply_u3(theta, phi, lam, target)
    
    # Apply the multi-controlled operation
    qml.ctrl(u3_op, control=controls, control_values=control_values)()

In [ ]:

def encoding_layer(inputs, nEncodings, nElements, qubits):
    """
    Superpixel Encoding Layer. Matches the 'encoding' function in SEQNN.ipynb.
    """
    loc = qubits[:6]    # 0-5
    target = qubits[6:9] # 6-8
    
    # Hadamard on location qubits
    for q in loc:
        qml.Hadamard(wires=q)

    # Encode data
    for nEncoding in range(nEncodings):
        for i in range(8):
            for j in range(8):
                # Format indices to binary control strings (e.g. 3 -> [0,1,1])
                row_bin = [int(x) for x in format(i, '03b')]
                col_bin = [int(x) for x in format(j, '03b')]
                ctrl_state = row_bin + col_bin 
                
                # Calculate input index
                base_idx = 64 * nElements * nEncoding + nElements * (8 * i + j)
                
                # Apply Controlled U3 for each of the 3 target qubits
                # Target 0
                apply_controlled_u3(
                    inputs[base_idx + 0], inputs[base_idx + 1], inputs[base_idx + 2],
                    target[0], loc, ctrl_state
                )
                # Target 1
                apply_controlled_u3(
                    inputs[base_idx + 3], inputs[base_idx + 4], inputs[base_idx + 5],
                    target[1], loc, ctrl_state
                )
                # Target 2
                apply_controlled_u3(
                    inputs[base_idx + 6], inputs[base_idx + 7], inputs[base_idx + 8],
                    target[2], loc, ctrl_state
                )
                
                # Entanglement
                qml.CZ(wires=[target[0], target[1]])
                qml.CZ(wires=[target[1], target[2]])
                qml.CZ(wires=[target[2], target[0]])

In [ ]:

def kernel_prepare(xloc, yloc, target, kernel, readout, params, ctrl):
    loc_states = [[0,0], [1,0], [0,1], [1,1]]
    for i, loc_state in enumerate(loc_states):
        # TFQ: controlled_by(yloc[0], xloc[0], target, kernel[0], control_values=...)
        controls = [yloc[0], xloc[0], target, kernel[0]]
        ctrl_values = loc_state + [1] + ctrl
        
        apply_controlled_u3(
            params[3*i+0], params[3*i+1], params[3*i+2],
            readout, controls, ctrl_values
        )

def conv_layer(xloc, yloc, target, kernel, readout, params):
    # Kernel 0 (ctrl=[0])
    kernel_prepare(xloc, yloc, target, kernel, readout, params[0:12], [0])
    # Kernel 1 (ctrl=[1])
    kernel_prepare(xloc, yloc, target, kernel, readout, params[12:24], [1])

def qdcnn_layer(params, nQconv, qubits):
    color = qubits[6:9]
    kernel = qubits[9:10]
    readout = qubits[10:12]
    
    # Hadamard on kernel
    qml.Hadamard(wires=kernel[0])
    
    for i in range(nQconv):
        base = i * 144
        # Layer sequence exactly matching TFQ code structure
        conv_layer(qubits[3:6], qubits[0:3], color[0], kernel, readout[0], params[base:base+24])
        conv_layer(qubits[4:6], qubits[1:3], readout[0], kernel, readout[1], params[base+24:base+48])
        
        conv_layer(qubits[3:6], qubits[0:3], color[1], kernel, readout[0], params[base+48:base+72])
        conv_layer(qubits[4:6], qubits[1:3], readout[0], kernel, readout[1], params[base+72:base+96])
        
        conv_layer(qubits[3:6], qubits[0:3], color[2], kernel, readout[0], params[base+96:base+120])
        conv_layer(qubits[4:6], qubits[1:3], readout[0], kernel, readout[1], params[base+120:base+144])

In [ ]:
# Define the QNode
# We use "diff_method='best'" which typically selects adjoint differentiation for lightning devices
@qml.qnode(dev, interface="torch", diff_method="best")
def seqnn_circuit(inputs, conv_params):
    qubits = list(range(N_QUBITS))
    
    # 1. Encoding
    encoding_layer(inputs, 1, 9, qubits)
    
    # 2. QDCNN
    qdcnn_layer(conv_params, 1, qubits)
    
    # 3. Measurement Optimization
    # The original paper measures 64 X-basis observables.
    # We rotate X -> Z using Hadamard and measure probabilities.
    # This captures the full state information needed to reconstruct the 64 values
    # without running the circuit 64 times.
    meas_qubits = [2, 5, 11, 9, 6, 7, 8] # Indices from readout function
    
    for q in meas_qubits:
        qml.Hadamard(wires=q)
        
    # Returning probabilities (dimension 2^7 = 128) contains all information 
    # to compute the 64 expectation values linearly.
    return qml.probs(wires=meas_qubits)

In [ ]:
# =============================================================================
# 2. PYTORCH MODEL LAYERS
# =============================================================================

class SuperpixelLayer(nn.Module):
    def __init__(self, nElements, nEncodings, pool, input_channels):
        super().__init__()
        self.nElements = nElements
        self.nEncodings = nEncodings
        self.pool = pool
        
        # Original TF: shape=(nEncodings, pool^2 * channels, nElements)
        # We use a ModuleList of Linear layers to replicate this structure
        self.input_dim = input_channels * (pool ** 2)
        self.projections = nn.ModuleList([
            nn.Linear(self.input_dim, nElements) for _ in range(nEncodings)
        ])
        
    def forward(self, x):
        # x shape: (Batch, 32, 32, C)
        b, h, w, c = x.shape
        x_perm = x.permute(0, 3, 1, 2) # (B, C, H, W)
        
        # Equivalent to tf.image.extract_patches with 'VALID' padding
        # Unfold creates patches of size (C * pool * pool)
        patches = torch.nn.functional.unfold(x_perm, kernel_size=self.pool, stride=self.pool)
        
        # Reshape to (B, Patches, Features) -> (B, 64, C*16)
        patches = patches.transpose(1, 2) 
        
        outputs = []
        for i, proj in enumerate(self.projections):
            # Apply dense layer to each patch
            out = torch.relu(proj(patches)) # (B, 64, nElements)
            outputs.append(out)
            
        if self.nEncodings > 1:
            outputs = torch.stack(outputs, dim=1)
        else:
            outputs = outputs[0].unsqueeze(1)
            
        # Flatten to (B, 576)
        return outputs.reshape(b, -1)

In [ ]:
class SEQNN(nn.Module):
    def __init__(self, n_classes, input_channels, nElements=9, nEncodings=1, pool=4):
        super().__init__()
        # 1. Classical Preprocessing
        self.superpixel = SuperpixelLayer(nElements, nEncodings, pool, input_channels)
        
        # 2. Quantum Layer Parameters
        # 144 parameters per QConv layer
        self.conv_params = nn.Parameter(torch.rand(144) * 2 * np.pi)
        
        # 3. Dense Classification Layer
        # The quantum layer outputs 128 probabilities (informationally complete for the 64 observables).
        # We allow the dense layer to learn the linear mapping from these probs to classes.
        self.dense = nn.Linear(128, n_classes) 
        
    def forward(self, x):
        # Preprocessing
        x = self.superpixel(x)
        
        # Quantum Circuit
        # We calculate probs for the batch. PennyLane handles batching automatically
        # if the device supports it (lightning.qubit does).
        x_q = seqnn_circuit(x, self.conv_params)
        
        # Cast to float32 (PennyLane often returns float64)
        x_q = x_q.float()
        
        # Classification
        x_out = self.dense(x_q)
        return torch.softmax(x_out, dim=1)

In [ ]:
# =============================================================================
# 3. TRAINING LOOP
# =============================================================================

def main():
    # Settings
    DATASET = 'cifar10' # Options: 'sat', 'lcz', 'overhead', 'cifar10'
    BATCH_SIZE = 32
    EPOCHS = 10
    LR = 0.01
    
    # 1. Load Data
    print(f"Loading {DATASET} dataset...")
    raw_loader = OriginalDataLoader(DATASET)
    
    # Convert to PyTorch Tensors
    train_x = torch.from_numpy(raw_loader.train_x).float()
    train_y = torch.from_numpy(raw_loader.train_y).float()
    test_x = torch.from_numpy(raw_loader.test_x).float()
    test_y = torch.from_numpy(raw_loader.test_y).float()
    
    # Create DataLoaders
    train_ds = TensorDataset(train_x, train_y)
    test_ds = TensorDataset(test_x, test_y)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE)
    
    # 2. Initialize Model
    n_classes = train_y.shape[1]
    n_channels = train_x.shape[-1]
    
    model = SEQNN(n_classes, n_channels)
    
    # Setup device (GPU if available)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    print(f"Model initialized on {device}")
    
    # 3. Optimizer & Loss
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()
    
    # 4. Training Loop
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        correct = 0
        total_samples = 0
        
        for batch_x, batch_y in train_dl:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_x)
            
            # Use argmax for labels in CrossEntropy
            loss = criterion(outputs, torch.argmax(batch_y, dim=1))
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item() * batch_x.size(0)
            pred = torch.argmax(outputs, dim=1)
            true = torch.argmax(batch_y, dim=1)
            correct += (pred == true).sum().item()
            total_samples += batch_x.size(0)
            
        # Evaluation
        train_acc = correct / total_samples
        print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {total_loss/total_samples:.4f} - Acc: {train_acc:.4f}")

    # 5. Final Test
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_x, batch_y in test_dl:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            pred = torch.argmax(outputs, dim=1)
            true = torch.argmax(batch_y, dim=1)
            correct += (pred == true).sum().item()
            total += batch_x.size(0)
            
    print(f"Final Test Accuracy: {correct/total:.4f}")

if __name__ == "__main__":
    main()